# Classification Results — Iris

**Este notebook NO entrena modelos.**

Solo lee los JSON generados por `Notebooks/Classification/` y crea las figuras
para el documento y la defensa.

Archivos esperados en `Notebooks/Results/metrics/`:
`svm_iris.json`, `rf_iris.json`, `xgb_iris.json`, `qsvm_iris.json`, `qnn_iris.json`, `vqc_iris.json`


## Cargar métricas


In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

cwd = Path.cwd()
if cwd.name == "Results":
    RESULTS_DIR = cwd
elif (cwd / "Notebooks" / "Results").exists():
    RESULTS_DIR = cwd / "Notebooks" / "Results"
else:
    RESULTS_DIR = cwd

FIGURES_DIR = RESULTS_DIR / "figures"
METRICS_DIR = RESULTS_DIR / "metrics"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED = {
    "SVM": "svm_iris.json",
    "Random Forest": "rf_iris.json",
    "XGBoost": "xgb_iris.json",
    "QSVM": "qsvm_iris.json",
    "QNN": "qnn_iris.json",
    "VQC": "vqc_iris.json",
}
ALL_MODELS = list(EXPECTED.keys())

results = {}
missing = []
for model, filename in EXPECTED.items():
    path = METRICS_DIR / filename
    if not path.exists():
        missing.append(str(path))
        continue
    results[model] = json.loads(path.read_text())

if missing:
    raise FileNotFoundError(
        "Faltan métricas. Ejecuta primero los notebooks de Classification.\n"
        + "\n".join(f" - {m}" for m in missing)
    )

print(f"Loaded {len(results)}/{len(EXPECTED)} from {METRICS_DIR}")
for m in ALL_MODELS:
    print(f"  {m}: test acc={results[m]['test']['accuracy']:.4f}")


## Tabla consolidada


In [ ]:
rows = []
for name in ALL_MODELS:
    r = results[name]
    for split in ("train", "test"):
        rows.append({
            "model": name,
            "family": r["family"],
            "split": split,
            **{k: r[split].get(k) for k in ["accuracy", "precision", "recall", "f1", "auc"]},
            "train_time_s": r["train_time_s"],
            "inference_time_s": r["inference_time_s"],
        })

df = pd.DataFrame(rows)
df_test = df[df["split"] == "test"].set_index("model").loc[ALL_MODELS].reset_index()
df_train = df[df["split"] == "train"].set_index("model").loc[ALL_MODELS].reset_index()

summary = df_test[
    ["model", "family", "accuracy", "precision", "recall", "f1", "auc", "train_time_s", "inference_time_s"]
].copy()
summary["acc_gap"] = (
    df_train.set_index("model").loc[summary["model"], "accuracy"].to_numpy(dtype=float)
    - summary["accuracy"].to_numpy(dtype=float)
)
print(summary.round(4).to_string(index=False))

summary_path = METRICS_DIR / "iris_classification_summary.csv"
summary.to_csv(summary_path, index=False)
print("\nSaved:", summary_path)


## Estilo y paleta

Convención: **clásico = azul oscuro / gris / negro**; **cuántico = azules claros**.


In [ ]:
# Paleta: classic oscuro, quantum claro
CLASSIC_COLORS = {
    "SVM": "#0B1F33",          # casi negro azulado
    "Random Forest": "#0E4A75", # navy UNIR
    "XGBoost": "#4A5560",       # gris oscuro
}
QUANTUM_COLORS = {
    "QSVM": "#7EC8E3",          # azul claro
    "QNN": "#A8D5E5",           # celeste
    "VQC": "#D6EEF5",           # azul muy claro
}
MODEL_COLORS = {**CLASSIC_COLORS, **QUANTUM_COLORS}

UNIR_NAVY = "#0E4A75"
UNIR_AZUL = "#219BC3"
UNIR_GRIS = "#858591"
UNIR_CLARO = "#C0C0C8"

plt.rcParams.update({
    "font.sans-serif": ["Arial", "DejaVu Sans", "Helvetica"],
    "axes.edgecolor": UNIR_CLARO,
    "grid.color": UNIR_CLARO,
    "grid.linestyle": "--",
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "figure.dpi": 150,
    "legend.fontsize": 9,
})

metric_cols = ["accuracy", "precision", "recall", "f1", "auc"]
metric_labels = ["Accuracy", "Precision", "Recall", "F1", "AUC"]


def savefig(name):
    path = FIGURES_DIR / name
    plt.savefig(path, bbox_inches="tight", facecolor="white")
    print("Saved:", path)
    plt.show()


def annotate_bars(ax, container, fmt="{:.2f}", dy=0.015, fontsize=7):
    for bar in container:
        h = bar.get_height()
        if np.isnan(h):
            continue
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            h + dy,
            fmt.format(h),
            ha="center",
            va="bottom",
            fontsize=fontsize,
            color=UNIR_GRIS,
        )


### 1. Métricas de test por modelo


In [ ]:
fig, ax = plt.subplots(figsize=(12, 5.5))
x = np.arange(len(metric_labels))
width = 0.13

for i, model in enumerate(ALL_MODELS):
    vals = df_test.set_index("model").loc[model, metric_cols].to_numpy(dtype=float)
    ax.bar(
        x + (i - 2.5) * width,
        vals,
        width,
        label=model,
        color=MODEL_COLORS[model],
        edgecolor="white",
        linewidth=0.4,
        zorder=3,
    )

ax.set_xticks(x)
ax.set_xticklabels(metric_labels)
ax.set_ylim(0, 1.18)
ax.set_ylabel("Score")
ax.set_title("Iris — Test metrics by model", color=UNIR_NAVY, pad=8)
ax.grid(axis="y", zorder=0)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# Leyenda fuera del área de datos
ax.legend(
    frameon=False,
    ncol=3,
    loc="upper center",
    bbox_to_anchor=(0.5, -0.12),
)
fig.subplots_adjust(bottom=0.22)
savefig("iris_test_metrics_by_model.png")


### 2. Train vs Test (Accuracy y F1)

Mismo color de familia; train = relleno sólido, test = relleno más claro / borde.


In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 5.2), sharey=True)

for ax, metric, title in zip(axs, ["accuracy", "f1"], ["Accuracy", "F1-score"]):
    x = np.arange(len(ALL_MODELS))
    width = 0.35
    train_vals = df_train.set_index("model").loc[ALL_MODELS, metric].to_numpy(dtype=float)
    test_vals = df_test.set_index("model").loc[ALL_MODELS, metric].to_numpy(dtype=float)
    colors = [MODEL_COLORS[m] for m in ALL_MODELS]

    ax.bar(
        x - width / 2,
        train_vals,
        width,
        color=colors,
        edgecolor="#111111",
        linewidth=0.6,
        label="Train",
        zorder=3,
    )
    ax.bar(
        x + width / 2,
        test_vals,
        width,
        color=colors,
        alpha=0.45,
        edgecolor="#111111",
        linewidth=0.6,
        label="Test",
        zorder=3,
    )
    ax.set_xticks(x)
    ax.set_xticklabels(ALL_MODELS, rotation=20, ha="right")
    ax.set_ylim(0, 1.18)
    ax.set_title(title, color=UNIR_NAVY)
    ax.grid(axis="y", zorder=0)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    if ax is axs[0]:
        ax.set_ylabel("Score")

# Una sola leyenda global debajo
handles, labels = axs[0].get_legend_handles_labels()
# Deduplicate Train/Test handles only
fig.legend(
    handles[:2],
    ["Train (opaco)", "Test (transparente)"],
    frameon=False,
    loc="upper center",
    bbox_to_anchor=(0.5, 0.02),
    ncol=2,
)
fig.suptitle("Iris — Train vs Test generalization", color=UNIR_NAVY, y=0.98)
fig.tight_layout(rect=[0, 0.08, 1, 0.95])
savefig("iris_train_vs_test.png")


### 3. Clásicos vs cuánticos (media en test)


In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 4.8))
x = np.arange(len(metric_labels))
width = 0.35
classical = np.nanmean(
    df_test[df_test["family"] == "classical"][metric_cols].to_numpy(dtype=float), axis=0
)
quantum = np.nanmean(
    df_test[df_test["family"] == "quantum"][metric_cols].to_numpy(dtype=float), axis=0
)
b1 = ax.bar(x - width / 2, classical, width, label="Classical (mean)", color="#0E4A75", zorder=3)
b2 = ax.bar(x + width / 2, quantum, width, label="Quantum (mean)", color="#A8D5E5",
            edgecolor="#0E4A75", linewidth=0.6, zorder=3)
annotate_bars(ax, b1)
annotate_bars(ax, b2)
ax.set_xticks(x)
ax.set_xticklabels(metric_labels)
ax.set_ylim(0, 1.25)
ax.set_ylabel("Score")
ax.set_title("Iris — Classical vs Quantum (test mean)", color=UNIR_NAVY, pad=10)
ax.legend(frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=2)
ax.grid(axis="y", zorder=0)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
fig.subplots_adjust(bottom=0.2)
savefig("iris_classical_vs_quantum.png")


### 4. Gap de generalización (train − test)


In [ ]:
gap_acc = (
    df_train.set_index("model").loc[ALL_MODELS, "accuracy"].to_numpy(dtype=float)
    - df_test.set_index("model").loc[ALL_MODELS, "accuracy"].to_numpy(dtype=float)
)
gap_f1 = (
    df_train.set_index("model").loc[ALL_MODELS, "f1"].to_numpy(dtype=float)
    - df_test.set_index("model").loc[ALL_MODELS, "f1"].to_numpy(dtype=float)
)

fig, ax = plt.subplots(figsize=(9.5, 4.8))
x = np.arange(len(ALL_MODELS))
width = 0.35
ax.bar(x - width / 2, gap_acc, width, label="Accuracy gap", color="#0B1F33", zorder=3)
ax.bar(x + width / 2, gap_f1, width, label="F1 gap", color="#A8D5E5",
       edgecolor="#0B1F33", linewidth=0.6, zorder=3)
ax.axhline(0, color=UNIR_GRIS, lw=1)
ax.set_xticks(x)
ax.set_xticklabels(ALL_MODELS, rotation=20, ha="right")
ax.set_ylabel("Train − Test")
ax.set_title("Iris — Generalization gap", color=UNIR_NAVY, pad=10)
ax.legend(frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.18), ncol=2)
ax.grid(axis="y", zorder=0)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
fig.subplots_adjust(bottom=0.25)
savefig("iris_generalization_gap.png")


### 5. Tiempos: training y prediction


In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 5))
x = np.arange(len(ALL_MODELS))
train_times = df_test.set_index("model").loc[ALL_MODELS, "train_time_s"].to_numpy(dtype=float)
pred_times = df_test.set_index("model").loc[ALL_MODELS, "inference_time_s"].to_numpy(dtype=float)
colors = [MODEL_COLORS[m] for m in ALL_MODELS]

axs[0].bar(x, train_times, color=colors, edgecolor="white", zorder=3)
axs[0].set_xticks(x)
axs[0].set_xticklabels(ALL_MODELS, rotation=20, ha="right")
axs[0].set_ylabel("Seconds (log)")
axs[0].set_title("Training time", color=UNIR_NAVY)
axs[0].set_yscale("log")
axs[0].grid(axis="y", zorder=0)
axs[0].spines["top"].set_visible(False)
axs[0].spines["right"].set_visible(False)

axs[1].bar(x, pred_times, color=colors, edgecolor="white", zorder=3)
axs[1].set_xticks(x)
axs[1].set_xticklabels(ALL_MODELS, rotation=20, ha="right")
axs[1].set_ylabel("Seconds (log)")
axs[1].set_title("Prediction time", color=UNIR_NAVY)
axs[1].set_yscale("log")
axs[1].grid(axis="y", zorder=0)
axs[1].spines["top"].set_visible(False)
axs[1].spines["right"].set_visible(False)

fig.suptitle("Iris — Computational cost", color=UNIR_NAVY, y=0.98)
fig.tight_layout(rect=[0, 0.03, 1, 0.95])
savefig("iris_runtimes.png")


### 6. Heatmap resumen (test)


In [ ]:
mat = df_test.set_index("model").loc[ALL_MODELS, metric_cols]
fig, ax = plt.subplots(figsize=(8.5, 4.6))
im = ax.imshow(mat.to_numpy(dtype=float), cmap="Blues", vmin=0.5, vmax=1.0, aspect="auto")
ax.set_xticks(range(len(metric_labels)))
ax.set_xticklabels(metric_labels)
ax.set_yticks(range(len(ALL_MODELS)))
ax.set_yticklabels(ALL_MODELS)
ax.set_title("Iris — Test metrics heatmap", color=UNIR_NAVY, pad=10)
for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
        val = mat.iloc[i, j]
        label = "—" if pd.isna(val) else f"{val:.2f}"
        ax.text(j, i, label, ha="center", va="center", color="black", fontsize=9)
cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.ax.tick_params(labelsize=8)
fig.tight_layout()
savefig("iris_test_heatmap.png")


### 7. Ranking de F1 en test

Útil en slides: una sola métrica, orden claro de mejor a peor.


In [ ]:
rank = df_test[["model", "family", "f1"]].sort_values("f1", ascending=True)
fig, ax = plt.subplots(figsize=(8, 4.5))
colors = [MODEL_COLORS[m] for m in rank["model"]]
bars = ax.barh(rank["model"], rank["f1"], color=colors, edgecolor="white", zorder=3)
for bar, val in zip(bars, rank["f1"]):
    ax.text(val + 0.01, bar.get_y() + bar.get_height() / 2, f"{val:.3f}",
            va="center", fontsize=8, color=UNIR_GRIS)
ax.set_xlim(0, 1.15)
ax.set_xlabel("F1 (test)")
ax.set_title("Iris — Test F1 ranking", color=UNIR_NAVY, pad=10)
ax.grid(axis="x", zorder=0)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
fig.tight_layout()
savefig("iris_f1_ranking.png")


### 8. Pendiente train → test (Accuracy)

Muestra de un vistazo quién generaliza y quién cae (p. ej. QSVM).


In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 5))
for model in ALL_MODELS:
    tr = float(df_train.set_index("model").loc[model, "accuracy"])
    te = float(df_test.set_index("model").loc[model, "accuracy"])
    color = MODEL_COLORS[model]
    ax.plot([0, 1], [tr, te], color=color, lw=2, zorder=2)
    ax.scatter([0, 1], [tr, te], color=color, s=60, zorder=3, edgecolor="white")
    ax.text(1.03, te, model, va="center", fontsize=9, color=color)

ax.set_xticks([0, 1])
ax.set_xticklabels(["Train", "Test"])
ax.set_xlim(-0.15, 1.45)
ax.set_ylim(0.6, 1.05)
ax.set_ylabel("Accuracy")
ax.set_title("Iris — Accuracy slope (train → test)", color=UNIR_NAVY, pad=10)
ax.grid(axis="y", zorder=0)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
fig.tight_layout()
savefig("iris_accuracy_slope.png")


### 9. Rendimiento vs coste (F1 test vs training time)

Para argumentar trade-off precisión/tiempo en la defensa.


In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 5))
for model in ALL_MODELS:
    f1 = float(df_test.set_index("model").loc[model, "f1"])
    t = float(df_test.set_index("model").loc[model, "train_time_s"])
    ax.scatter(
        t, f1,
        s=120,
        color=MODEL_COLORS[model],
        edgecolor="#111111",
        linewidth=0.6,
        zorder=3,
        label=model,
    )
    ax.text(t * 1.05 if t > 0 else 0.01, f1 + 0.01, model, fontsize=8, color=MODEL_COLORS[model])

ax.set_xscale("log")
ax.set_xlabel("Training time (s, log)")
ax.set_ylabel("F1 (test)")
ax.set_ylim(0.6, 1.05)
ax.set_title("Iris — Performance vs training cost", color=UNIR_NAVY, pad=10)
ax.grid(True, zorder=0, which="both")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.legend(frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.14), ncol=3)
fig.subplots_adjust(bottom=0.22)
savefig("iris_performance_vs_cost.png")


### 10. Rendimiento vs coste de predicción

Complementa el plot de training: en modelos cuánticos el coste de **predecir**
(kernel / circuitos) suele ser el argumento operativo más relevante.


In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 5))
for model in ALL_MODELS:
    f1 = float(df_test.set_index("model").loc[model, "f1"])
    t = float(df_test.set_index("model").loc[model, "inference_time_s"])
    # evitar log(0)
    t_plot = max(t, 1e-6)
    ax.scatter(
        t_plot,
        f1,
        s=120,
        color=MODEL_COLORS[model],
        edgecolor="#111111",
        linewidth=0.6,
        zorder=3,
        label=model,
    )
    ax.text(t_plot * 1.08, f1 + 0.01, model, fontsize=8, color=MODEL_COLORS[model])

ax.set_xscale("log")
ax.set_xlabel("Prediction time (s, log)")
ax.set_ylabel("F1 (test)")
ax.set_ylim(0.6, 1.05)
ax.set_title("Iris — Performance vs prediction cost", color=UNIR_NAVY, pad=10)
ax.grid(True, zorder=0, which="both")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.legend(frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.14), ncol=3)
fig.subplots_adjust(bottom=0.22)
savefig("iris_performance_vs_prediction_cost.png")


## Archivos generados


In [ ]:
print("Metrics JSON:")
for p in sorted(METRICS_DIR.glob("*.json")):
    print(" -", p)
print("\nFigures:")
for p in sorted(FIGURES_DIR.glob("iris_*.png")):
    print(" -", p)
